# Colab CNN Benchmark On Predefined CSV Splits

This notebook runs the CNN baseline on the shared `train`, `eval`/validation, and `test` CSV files. It uses the benchmark config from the repo so the dataset paths, seed, metrics, and threshold rule stay consistent with future DNABERT2 and iPro-MP runs.

## 1. Set Up SeqTrainer In Colab

This clones the working branch and installs SeqTrainer. After this branch is merged, change `BRANCH` to `dev`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-cnn-baseline-reproduction"
REPO_DIR = Path("/content/SeqTrainer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[torch]"], check=True)
print("SeqTrainer installed")

## 2. Mount Drive And Copy The Three Dataset Files

The benchmark config expects the files under `data/promoter_classification/`. This cell copies them from Drive into that local Colab repo path.

In [ ]:
from pathlib import Path
import shutil
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("google.colab is not available. Assuming dataset files are already local or in the repo zip.")

LOCAL_DATA_DIR = Path("data/promoter_classification")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

# The verified Drive location is AIxBio / Promoter Classification / Data.
# Colab usually exposes My Drive as /content/drive/MyDrive, but this helper also
# checks the older /content/drive/My Drive spelling and searches inside AIxBio.
expected_relative_dir = Path("AIxBio") / "Promoter Classification" / "Data"
drive_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/My Drive"),
    Path("/content/drive/Shareddrives"),
]


def has_all_split_files(directory: Path) -> bool:
    return all((directory / file_name).exists() for file_name in split_file_names.values())


def find_drive_data_dir() -> Path | None:
    candidates = [root / expected_relative_dir for root in drive_roots]
    print("Checking expected Drive paths:")
    for candidate in candidates:
        print(f"  - {candidate}")
        if has_all_split_files(candidate):
            return candidate

    # Keep this search narrow so Colab does not scan the whole Drive unnecessarily.
    search_roots = []
    for root in drive_roots:
        aixbio = root / "AIxBio"
        if aixbio.exists():
            search_roots.append(aixbio)
    for root in search_roots:
        print(f"Searching for train CSV under: {root}")
        for train_path in root.rglob(split_file_names["train"]):
            candidate = train_path.parent
            if has_all_split_files(candidate):
                return candidate
    return None


def copy_from_drive(data_dir: Path) -> bool:
    for split, file_name in split_file_names.items():
        source = data_dir / file_name
        target = LOCAL_DATA_DIR / file_name
        shutil.copy2(source, target)
        print(f"Copied {split}: {target}")
    return True


def extract_from_repo_zip() -> bool:
    zip_path = Path("data/data_DNABERT/promoter_classification_DNABERT.zip")
    if not zip_path.exists():
        return False

    print(f"Drive CSVs were not found locally. Extracting from repo zip: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        members = set(zf.namelist())
        for split, file_name in split_file_names.items():
            if file_name not in members:
                raise FileNotFoundError(f"{file_name} is missing from {zip_path}")
            target = LOCAL_DATA_DIR / file_name
            with zf.open(file_name) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)
            print(f"Extracted {split}: {target}")
    return True


drive_data_dir = find_drive_data_dir()
if drive_data_dir is not None:
    print(f"Using Drive data directory: {drive_data_dir}")
    copy_from_drive(drive_data_dir)
elif all((LOCAL_DATA_DIR / file_name).exists() for file_name in split_file_names.values()):
    print(f"Using existing local CSV files in: {LOCAL_DATA_DIR}")
elif extract_from_repo_zip():
    print(f"Using CSV files extracted into: {LOCAL_DATA_DIR}")
else:
    raise FileNotFoundError(
        "Could not find the promoter CSV files in Drive or in the repo zip. "
        "Expected Drive folder: /content/drive/MyDrive/AIxBio/Promoter Classification/Data"
    )

## 3. Verify The Config And Dataset Splits

This checks that Colab is using the same split files from the benchmark config. The `eval` file is used as validation for threshold selection and model selection.

In [ ]:
import pandas as pd
from seqtrainer.benchmarks import load_benchmark_config

CONFIG_PATH = Path("config-examples/benchmarks/cnn.toml")
config = load_benchmark_config(CONFIG_PATH)

print("Dataset:", config.dataset.name)
print("Source accession:", config.dataset.source_accession)
print("Split strategy:", config.split.strategy)
print("Primary metric:", config.evaluation.primary_metric)

for split, csv_path in config.dataset.split_files.items():
    frame = pd.read_csv(csv_path)
    label_counts = frame[config.dataset.label_field].value_counts().sort_index().to_dict()
    sequence_lengths = frame[config.dataset.sequence_field].astype(str).str.len()
    print(
        f"{split}: rows={len(frame)} labels={label_counts} "
        f"length_min={sequence_lengths.min()} length_max={sequence_lengths.max()} "
        f"length_mean={sequence_lengths.mean():.1f}"
    )

## 4. Run The CNN Benchmark

The model trains on `train`, chooses the probability threshold using validation MCC on `eval`, and reports final metrics on `test` using that validation-selected threshold.

In [ ]:
import torch
from seqtrainer.torch.cnn_baseline import CnnCsvSplitConfig, run_cnn_csv_splits

CYCLES = 10
BATCH_SIZE = config.training.batch_size or 16
SEQUENCE_LENGTH = config.preprocessing.sequence_length or 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

run_config = CnnCsvSplitConfig(
    train_csv=config.dataset.split_files["train"],
    validation_csv=config.dataset.split_files["validation"],
    test_csv=config.dataset.split_files["test"],
    output_dir="outputs/cnn_csv_colab",
    dataset_name=config.dataset.name,
    source_accession=config.dataset.source_accession,
    source_url=config.dataset.source_url,
    sequence_field=config.dataset.sequence_field,
    label_field=config.dataset.label_field,
    sequence_length=SEQUENCE_LENGTH,
    seed=config.experiment.seed,
    batch_size=BATCH_SIZE,
    cycles=CYCLES,
    learning_rate=config.training.learning_rate or 1e-3,
    device=DEVICE,
)

result = run_cnn_csv_splits(run_config)
print("Output directory:", result.output_dir)
print("Validation-selected threshold:", result.manifest["threshold_selection"]["threshold"])

## 5. Inspect Metrics And Training History

The saved artifacts are the benchmark output we can compare later against DNABERT2 and iPro-MP.

In [ ]:
output_dir = Path(result.output_dir)

metrics_df = pd.read_csv(output_dir / "metrics.csv")
history_df = pd.read_csv(output_dir / "history.csv")

display(metrics_df)
display(history_df.tail())

## 6. Optional: Save Outputs Back To Drive

Keep this enabled if you want the metrics and predictions to persist after the Colab runtime disconnects.

In [ ]:
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Outputs/cnn_csv_colab")

if SAVE_TO_DRIVE:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for file_name in ["config.json", "manifest.json", "metrics.json", "metrics.csv", "history.csv", "predictions.csv"]:
        shutil.copy2(output_dir / file_name, DRIVE_OUTPUT_DIR / file_name)
    print("Saved outputs to:", DRIVE_OUTPUT_DIR)